In [1]:
# https://qwen.readthedocs.io/en/latest/framework/LlamaIndex.html

from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def completion_to_prompt(completion):
    return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"


def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt


def setup(model, tokenizer):
    Settings.llm = HuggingFaceLLM(
        model_name=model, # "Qwen/Qwen3-235B-A22B-Instruct-2507",  # "Qwen/Qwen2.5-7B-Instruct",
        tokenizer_name=tokenizer, # "Qwen/Qwen3-235B-A22B-Instruct-2507",  # "Qwen/Qwen2.5-7B-Instruct",
        context_window=30000,
        max_new_tokens=10000,
        generate_kwargs={
            "temperature": 0.7,
            "top_k": 20,
            "top_p": 0.8,
        },  # {"temperature": 0.7, "top_k": 50, "top_p": 0.95},
        messages_to_prompt=messages_to_prompt,
        completion_to_prompt=completion_to_prompt,
        device_map="auto",
    )

def create_save_vsidx(save_dir, embed_model, input_dir = None, input_files = None):
    Settings.embed_model = HuggingFaceEmbedding(
        model_name=embed_model, # "BAAI/bge-multilingual-gemma2"  # "BAAI/bge-base-en-v1.5"
    )

    Settings.transformations = [SentenceSplitter(chunk_size=1024)]

    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

    if input_dir is not None:
        documents = SimpleDirectoryReader(
            input_dir=input_dir
        ).load_data()
    elif input_files is not None:
        documents = SimpleDirectoryReader(
            input_files=input_files
        ).load_data()
        
    Settings.index = VectorStoreIndex.from_documents(
        documents,
        embed_model=Settings.embed_model,
        transformations=Settings.transformations,
    )
    Settings.index.storage_context.persist(persist_dir=save_dir)
    
def load_vsidx(save_dir):
    from llama_index import StorageContext, load_index_from_storage
    storage_context = StorageContext.from_defaults(persist_dir=save_dir)

    # optional service context
    Settings.index = load_index_from_storage(storage_context)



In [2]:
create_save_vsidx(
    save_dir="syuu",
    embed_model="BAAI/bge-m3", #"BAAI/bge-multilingual-gemma2",
    input_dir="../translateDirFiles/資治通鑑" )



In [3]:
#setup(
#    model = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
#    tokenizer = "Qwen/Qwen3-14B-FP8")

setup(
    model = "Qwen/Qwen3-14B-FP8",
    tokenizer = "Qwen/Qwen3-0.6B-FP8")

2026-03-28 08:30:26,547 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-28 08:30:47,851 - WARNING - The model `Qwen/Qwen3-14B-FP8` and tokenizer `Qwen/Qwen3-0.6B-FP8` are different, please ensure that they are compatible.


In [4]:
def query(query_text):
    query_engine = Settings.index.as_query_engine(streaming=True, similarity_top_k=10)
    r = query_engine.query(query_text)
    r.print_response_stream()
    
query("匈奴と漢の関係について教えてください。")
    

<think>
Okay, I need to explain the relationship between the Xiongnu (匈奴) and the Han Dynasty (漢) based on the provided context. Let me start by going through the given texts to gather relevant information.

First, in the 014_漢紀_06.txt, there's a mention of the Han emperor negotiating with the Xiongnu leader, suggesting a truce and mutual respect. The emperor sends a message acknowledging the Xiongnu's desire to resolve past issues and restore old agreements. This shows periods of diplomacy and attempts at peaceful relations.

Looking at 019_漢紀_11.txt, there's a story about the Xiongnu leader Huo Hunye (渾邪王) defecting to the Han. The Han sends General Wei Qing (驃騎將軍) to receive them, leading to a significant number of Xiongnu people defecting. This indicates military confrontations and instances where the Xiongnu surrendered to the Han.

In 017_漢紀_09.txt, the Han considers a truce with the Xiongnu, with debates among officials about whether to accept the truce or attack. This shows the

In [5]:
query("高車の動向について教えてください。")

<think>
Okay, I need to answer the user's question about the movements of the Gaoche (高車). Let me look through the provided context information to find relevant mentions of Gaoche.

First, in the file 149_梁紀_05.txt, there's a detailed account. It starts by mentioning that after the death of Gaoche King Miroutu, his people all went to Yeta. Later, Yeta sent Miroutu's brother Yifú to lead the remaining people back to their country. Yifú defeated the Rouran Khan Bolomen, and Bolomen with ten tribes went to Liangzhou to surrender to Wei. Then there's a discussion about whether to accept the surrender of Rouran, with suggestions to keep both Rouran leaders, placing Anagu in the east and Bolomen in the west. Also, there's mention of the location of Xihai (西海) and military strategies involving the Gaoche.

Another part in the same file talks about Gaoche King Yifú sending envoys to offer tribute to Wei. In the file 152_梁紀_08.txt, there's a mention of Gaoche in the context of military strategi